# Tarea Opcional 2 

## Enunciado

**1.** *(0,4 décimas para la I1)* Un estanque rígido contiene un fluido, del cual lo único que se sabe es que sus propiedades críticas son: $T_c = 591,8~\text{K}$, $P_c = 41,09~\text{bar}$ y $\omega = 0,264$. Para este fluido, se puede considerar un $C_p^{GI} = 108,8~\text{J/mol}\cdot\text{K}$. Inicialmente, el fluido se encuentra a $313.15~\text{K}$ y $2~\text{bar}$. Posteriormente, éste se calienta hasta que se alcanza una presión interna de $100~\text{bar}$. Usando la ecuación de *Redlich-Kwong* responda:

**a)** ¿En qué fase se encuentra el fluido según la ecuación de estado?

**b)** La presión de vapor (en $\text{bar}$) a la temperatura inicial del estanque. ¿Tiene sentido con la respuesta del punto anterior?

**c)** La temperatura final ($\text{K}$)

**d)** El calor transferido hacia el estanque por cada mol de fluido ($\text{J/mol}$)

```{admonition} Importante
:class: important
Debe demostrar el resultado final paso a paso. En este curso el procedimiento y que demuestren conocimiento en la búsqueda del resultado final es lo más importante. Solamente agregar números a una fórmula sin explicación, no se considera una resolución aceptable. Pueden entregar la tarea escrita a mano y escaneada o en computador, en general, como les sea más cómodo, o de la forma que les quite menos tiempo. Lo importante es que se entienda el procedimiento.
```

{ref}`Ver solución <t:2024-1:tarea2>`

<br/><br/>
<br/><br/>
<br/><br/>
<br/><br/>
<br/><br/>
<br/><br/>
<br/><br/>
<br/><br/>
<br/><br/>
<br/><br/>
<br/><br/>
<br/><br/>

(t:2024-1:tarea2)=
## Solución

In [1]:
# Paquetes de python utilizados para la resolución de esta tarea
import handcalcs.render
from handcalcs import handcalc
from numpy import sqrt, cos, arccos, pi, exp, log
from scipy.optimize import fsolve

### Inciso A

Siempre que tengamos una expresión cúbica para las ecuaciones de estado que sea del siguiente tipo:

$$
x^3 + a_2x_2^2 + a_1x_1 + a_0 = 0
$$

Se pueden obtener las raices y los parámetros utilizando el método de cardano. Para *Redlich-Kwong*, tenemos los siguientes parámetros.

$$
a_0 = -\frac{ab}{P\sqrt{T}} \hspace{1cm} a_1 = \frac{1}{P}\left(\frac{a}{\sqrt{T}}-bRT-Pb^2\right) \hspace{1cm} a_2 = -\frac{RT}{P}
$$

Recordando que los parámetros $a$ y $b$ para esta ecuación de estado son:

$$
a =0.42748\frac{R^2T_c^{2.5}}{P_c} \hspace{1cm} b = 0.08664\frac{RT_c}{P_c}
$$

Primero, se determina el discriminante ($q^3+r^2$) para saber si tenemos más de una raíz. En donde:

$$
q := \frac{3a_1-a_2^2}{9} \hspace{1cm} r:= \frac{a_2\left(9a_1-2a_2^2\right)-27a_0}{54}
$$

In [2]:
%%render long params
R = 8.314 #J/mol$\cdot$K
P_c = 41.09e5 #Pa
T_c = 591.8 #K
omega = 0.264
P = 2e5 #Pa
T = 313.15 #K

<IPython.core.display.Latex object>

In [3]:
%%render long sci_not 3
a = 0.42748 * (R**2*T_c**(2.5))/(P_c)
b = 0.08664 * (R*T_c)/P_c
a_0 = -(a*b/(P*sqrt(T)))
a_1 = 1/P*(a/sqrt(T)-b*R*T-P*b**2)
a_2 = -(R*T)/P
q = (3*a_1-a_2**2)/9
r = (a_2*(9*a_1-2*a_2**2)-27*a_0)/54
discriminante = q**3+r**2

<IPython.core.display.Latex object>

Ya que el discriminante es negativo, tenemos tres raices para nuestro polinomio.

$$
x_1 = 2\sqrt{-q}\cos{\left(\frac{\theta}{3}\right)}-\frac{a_2}{3}
$$

$$
x_2 = -2\sqrt{-q}\cos{\left(\frac{\theta-\pi}{3}\right)}-\frac{a_2}{3}
$$

$$
x_3 = -2\sqrt{-q}\cos{\left(\frac{\theta+\pi}{3}\right)}-\frac{a_2}{3}
$$

$$
\theta = \text{arcos}\left(\frac{r}{\sqrt{-q^3}}\right)
$$

Calculamos las raices, en donde el valor más bajo de volumen corresponderá al volumen de saturación asociado a la fase líquida, mientras que el mayor valor corresponderá a la fase gaseosa.

In [4]:
%%render long sci_not 5
theta = arccos(r/sqrt(-q**3))
x_1 = 2*sqrt(-q)*cos(theta/3)-a_2/3
x_2 = -2*sqrt(-q)*cos((theta-pi)/3)-a_2/3
x_3 = -2*sqrt(-q)*cos((theta+pi)/3)-a_2/3

<IPython.core.display.Latex object>

De esta manera, se obtiene que $V_{sat}^L = 1.25287\times10^{-4}~\text{m}^3\text{/mol}$ y $V_{sat}^V = 1.16633\times10^{-2}~\text{m}^3\text{/mol}$. Luego se utiliza el coeficiente de fugacidad para ubicar la fase en que nos encontramos.

$$
\ln\varphi = \frac{1}{RT}\int_{\infty}^{V}\left(\frac{RT}{V}-P\right)dV + (Z-1)-\ln Z
$$

Podemos entonces reemplazar la ecuación de estado tipo *Redlich-Kwong* en la expresión anterior, recordando que la expresión para esta *EOS* es:

$$
P = \frac{RT}{V-b} - \frac{aT^{-0.5}}{V(V+b)}
$$

Luego la integral nos queda de la siguiente manera:

$$
\int_{\infty}^{V}\left(\frac{RT}{V}-\frac{RT}{V-b} + \frac{aT^{-0.5}}{V(V+b)}\right)dV
$$

Separando la integral y calculando para los límites de integración.

$$
\int_{\infty}^{V}\left(\frac{RT}{V}-\frac{RT}{V-b}\right)dV = RT\ln\left(\frac{V}{V-b}\right)
$$

$$
\int_{\infty}^{V}\left(\frac{aT^{-0.5}}{V(V+b)}\right)dV = \frac{a}{\sqrt{T}b}\ln\left(\frac{V}{V+b}\right)
$$

Luego nos queda como

$$
\ln\varphi = \ln\left(\frac{V}{V-b}\right)+\frac{a}{\sqrt{T}bRT}\ln\left(\frac{V}{V+b}\right) + (Z-1)-\ln Z
$$

Luego calculamos para los volúmenes en ambos estados.

In [5]:
%%render long sci_not 5
V_L = x_2 #m$^3$/mol
V_V = x_1 #m$^3$/mol
Z_L = (P*V_L)/(R*T)
Z_V = (P*V_V)/(R*T)
phi_L = exp(log(V_L/(V_L-b))+a/(sqrt(T)*b*R*T)*log(V_L/(V_L+b)) + (Z_L-1) - log(Z_L))
phi_V = exp(log(V_V/(V_V-b))+a/(sqrt(T)*b*R*T)*log(V_V/(V_V+b)) + (Z_V-1) - log(Z_V))

<IPython.core.display.Latex object>

Ya que $\varphi_L < \varphi_V$, entonces nos encontramos en fase líquida.

### Inciso B

Podemos encontrar la presión de saturación cuando $\varphi_L = \varphi_V$.

In [6]:
# Definimos Redlich-Kwong
def redlich_kwong_phis(T, P, R, T_c, P_c):
    a = 0.42748 * (R**2*T_c**(2.5))/(P_c)
    b = 0.08664 * (R*T_c)/P_c
    a_0 = -(a*b/(P*sqrt(T)))
    a_1 = 1/P*(a/sqrt(T)-b*R*T-P*b**2)
    a_2 = -(R*T)/P
    q = (3*a_1-a_2**2)/9
    r = (a_2*(9*a_1-2*a_2**2)-27*a_0)/54
    # Calcular los valores de phi
    theta = arccos(r/sqrt(-q**3))
    V_V = 2*sqrt(-q)*cos(theta/3)-a_2/3
    V_L = -2*sqrt(-q)*cos((theta-pi)/3)-a_2/3
    Z_L = (P*V_L)/(R*T)
    Z_V = (P*V_V)/(R*T)
    phi_L = exp(log(V_L/(V_L-b))+a/(sqrt(T)*b*R*T)*log(V_L/(V_L+b)) + (Z_L-1) - log(Z_L))
    phi_V = exp(log(V_V/(V_V-b))+a/(sqrt(T)*b*R*T)*log(V_V/(V_V+b)) + (Z_V-1) - log(Z_V)) 
    phi = (phi_L, phi_V)
    return phi

# Función objetivo
def p_sat(T, P, R, T_c, P_c):
    def objective(P_sat):
        phi = redlich_kwong_phis(T=T, P=P_sat, R=R, T_c=T_c, P_c=P_c)
        return float(phi[0]) - float(phi[1])
    # Valor inicial
    P_guess = P/10
    # Encontrar valor óptimo
    P_sat = fsolve(objective, P_guess)[0]
    return P_sat

p_sat = p_sat(T, P, R, T_c, P_c)

C:\Users\igtab\AppData\Local\Temp\ipykernel_18824\2545850808.py:25: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  return float(phi[0]) - float(phi[1])


In [7]:
%%render params
p_sat # Pa

<IPython.core.display.Latex object>

Al parecer, este resultado es consistente con lo encontrado en el inciso anterior, ya que en un diagrama P vs V cualquiera, el disminuir la presión iremos acercándonos al punto de saturación del fluido dada una temperatura constante. Esto se corrobora ya que $2~\text{bar}>0.19686~\text{bar}$.

*Para una resolución más simple de este problema recomendamos utilizar Excel*.

### Inciso C

Ya que tenemos la presión ($P=100~\text{bar}$) el sistema en su estado final, podemos utilizar la ecuación de *Redlich-Kwong* para despejar $T$, ya que contamos con $V_L = 1.25287\times10^{-4}$.

In [8]:
def redlichkwong_eos(P, V):
    def eos(T):
        P_c = 41.09e5 #Pa
        T_c = 591.8 #K
        a = 0.42748 * (R**2*T_c**(2.5))/(P_c)
        b = 0.08664 * (R*T_c)/P_c        
        return (R*T/(V-b))-((a*T**(-0.5))/(V*(V+b)))-P
    # Solucion ideal como condicion inicial
    T_inicial = V*P/R
    # Resolver numericamente
    sol = fsolve(eos, x0=T_inicial)  
    return sol[0]

V_L = 1.25287e-4
P = 100e5
T_final = redlichkwong_eos(P,V_L)

In [9]:
%%render params
T_final #K

<IPython.core.display.Latex object>

### Inciso D 

Para calcular $\Delta U$, necesitamos hacer uso de las propiedades residuales. Tenemos que para $U^R$,

$$
U^R = H^R - P\cdot V^R
$$

Para el término de entalpía residual, necesitamos nuestra ecuación de estado

$$
H^R = PV - RT + \int_{\infty}^V\left[T\left(\frac{\partial P}{\partial T}\right)_V-P\right]dV
$$

Primero

$$
\left(\frac{\partial}{\partial T}\left(\frac{RT}{V-b} - \frac{a}{\sqrt{T}V(V+b)}\right)\right)_V = \frac{R}{V-b} + \frac{a}{2V(V+b)T^{3/2}}
$$

Luego la integral nos queda de la siguiente manera

$$
\int_{\infty}^V\left[\frac{RT}{V-b} + \frac{a}{2V(V+b)\sqrt{T}}  - \frac{RT}{V-b} + \frac{a}{V(V+b)\sqrt{T}}\right]dV = \int_{\infty}^V\left[\frac{3a}{2V(V+b)\sqrt{T}}\right]dV
$$

Calculando la integral y evaluando en los límites de integración llegamos al siguiente resultado:

$$
\int_{\infty}^V\left[\frac{3a}{2V(V+b)\sqrt{T}}\right]dV = \frac{3a}{2\sqrt{T}b}\ln\left(\frac{V}{V+b}\right)
$$

Luego nos queda la siguiente expresión para la entalpía de exceso.

$$
H^R = RT\left(Z-1\right) + \frac{3a}{2\sqrt{T}b}\ln\left(\frac{V}{V+b}\right)
$$

El volumen residual será simplemente la resta del valor obtenido mediante la ecuación de estado para $V_L$ y el valor ideal.

$$
V^R = V_L - V_L^{GI} = V_L - \frac{RT}{P}
$$

Se procede a calcular los valores residuales para el estado incial ($1$) y final ($2$).

In [10]:
%%render sci_not params
P_1 = 2e5 #Pa
P_2= 100e5 #Pa
T_1 = 313.15 #K
T_2 = T_final #K
V_L = 1.25287e-4 #m$^3$/mol
Z_1 = (V_L*P_1)/(R*T_1)
Z_2 = (V_L*P_2)/(R*T_2)

<IPython.core.display.Latex object>

In [11]:
%%render long sci_not
## Estado incial:
H_R1 = R*T_1*(Z_1-1) + (3*a)/(2*sqrt(T_1)*b)*log(V_L/(V_L+b))
V_R1 = V_L - (R*T_1)/P_1
U_R1 = H_R1 - P_1*V_R1
## Estado final:
H_R2 = R*T_2*(Z_2-1) + (3*a)/(2*sqrt(T_2)*b)*log(V_L/(V_L+b))
V_R2 = V_L - (R*T_2)/P_2
U_R2 = H_R2 - P_2*V_R2

<IPython.core.display.Latex object>

Por último, sabemos que 

$$
\Delta U = \Delta U^{GI} + \Delta U^R
$$

Donde

$$
\Delta U^{GI} = \left(C_p^{GI}-R\right)\Delta T
$$

In [12]:
%%render long
c_P = 108.8 #J/mol$\cdot$K
U_GI = (c_P - R)*(T_2-T_1) #J/mol
U = U_GI + (U_R2-U_R1) #J/mol

<IPython.core.display.Latex object>

Ya que nos encontramos en un estanque rígido sin término de trabajo $Q=U$. Por lo tanto el calor transferido al estanque por cada mol de fluido es $Q=2519.611~\text{J/mol}$.